In [1]:
import data.breathe_data as bd
import data.helpers as dh
import src.models.helpers as mh
from plotly.subplots import make_subplots
import datetime
import data.cfr_data_19_23 as cfrd
import pandas as pd
import cfr.cfr_viz_helpers as vh
import plotly.express as px
import plotly.graph_objs as go
from scipy.stats import spearmanr, permutation_test

In [2]:
df = bd.load_meas_from_excel(
    # "infer_all_19_data_with_best_FEV1",
    "pppfev1_ppfev1st_ppfev1ft_IV_19_plus1820_assoc_p(D|M)",
    study_folder="CFR",
    str_cols_to_arrays=[
        "Airway resistance (%)",
        "P(HFEV1|FEF2575, bFEV1, FEV1)",
        "P(HFEV1|bFEV1, FEV1)",
        "P(HFEV1|FEV1)",
    ],
    use_csv=True,
    bypass_sanity_checks=True,
)

In [4]:
df.columns

Index(['ID', 'Age', 'Height', 'best FEV1', 'Sex', 'Date Recorded', 'ecFEV1',
       'ecFEF2575', 'ecFEF2575%ecFEV1', 'Predicted FEV1', 'ecFEV1 % Predicted',
       'FEV1 % Predicted', 'best FEV1 old', 'idx FEV1', 'idx FEF2575%FEV1',
       'idx best FEV1', 'P(HFEV1|FEF2575, bFEV1, FEV1)', 'P(HFEV1|FEV1)',
       'Airway resistance (%)', 'P(HFEV1|bFEV1, FEV1)', 'P(HFEV1|bFEV1)',
       'mean FEV1PersPred', 'mean FEV1PredFT', 'mean FEV1PredST',
       'FEV1%PersPred', 'FEV1%PredFT', 'FEV1%PredST', 'pppFEV1 - ppFEV1ST',
       'ppFEV1FT - ppFEV1ST', 'P(FEV1|M)', 'P(FEF2575%FEV1|M, FEV1)',
       'P(bFEV1|M, FEV1, FEF2575%FEV1)', 'P(D|M)', 'IVs', 'IV days'],
      dtype='object')

In [46]:
diff_col = "ppFEV1FT - ppFEV1ST"

dftmp = df.copy()

# Filter percentile of certain population
prctile = 90
t = dftmp[diff_col].abs().quantile(prctile / 100)
print(f"{prctile}th percentile of absolute difference: {t}")

dftmp = dftmp[dftmp[diff_col].abs() > t]

# Filter by severity level
mask_mild = dftmp["FEV1%PredST"] >= 70
mask_moderate = (dftmp["FEV1%PredST"] >= 40) & (dftmp["FEV1%PredST"] < 70)
mask_severe = 40 > dftmp["FEV1%PredST"]
dftmp = dftmp[mask_mild]
# dftmp = dftmp[mask_moderate]
# dftmp = dftmp[mask_severe]

print(dftmp.shape)

fig = px.scatter(dftmp, diff_col, "IV days")

fig.show()

90th percentile of absolute difference: 1.8699132261817673
(188, 35)


In [49]:
df[df["IV days"] == 173]

,ID,Age,Height,best FEV1,Sex,Date Recorded,ecFEV1,ecFEF2575,ecFEF2575%ecFEV1,Predicted FEV1,...,FEV1%PredFT,FEV1%PredST,pppFEV1 - ppFEV1ST,ppFEV1FT - ppFEV1ST,P(FEV1|M),"P(FEF2575%FEV1|M, FEV1)","P(bFEV1|M, FEV1, FEF2575%FEV1)",P(D|M),IVs,IV days
154,B163943,37,179,4.21,Male,2019-01-01,3.75,3.27,87.199999,4.372914,...,80.205781,84.160162,-3.756488,-3.954382,0.01068,0.031161,0.008484,0.000003,3.666667,173.0
